# Intelligent I/O and Storage Scheduling

This notebook simulates an ML-based disk scheduling algorithm. We generate synthetic I/O traces, train a machine learning model to predict request service times (based on seek distance and I/O size), and compare an ML-augmented scheduler against a standard First-Come-First-Serve (FCFS) approach.

**GitHub Repository:** [https://github.com/yourusername/Intelligent-IO-Scheduling](https://github.com/yourusername/Intelligent-IO-Scheduling)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

## 1. Synthetic I/O Trace Generation
We generate synthetic I/O requests with Logical Block Address (LBA), data size, and calculate a 'true' latency based on a simulated disk model (seek time + transfer time).

In [ ]:
def generate_io_traces(num_requests=10000, max_lba=1000000, max_size_kb=1024):
    current_lba = 0
    data = []
    for _ in range(num_requests):
        lba = np.random.randint(0, max_lba)
        size = np.random.randint(4, max_size_kb)
        is_write = np.random.choice([0, 1])
        
        # Simulate physical disk latency
        seek_distance = abs(lba - current_lba)
        seek_time = 2.0 + (seek_distance / max_lba) * 8.0  # Base 2ms + up to 8ms seek penalty
        transfer_time = (size / 1024.0) * 1.5  # Assume 1.5ms per MB transfer time
        
        # Add some random noise to simulate OS/bus delays or disk glass effects
        noise = np.random.normal(0, 0.5)
        latency = seek_time + transfer_time + noise
        latency = max(0.5, latency) # Minimum hardware latency bound
        
        data.append({
            'lba': lba,
            'size_kb': size,
            'is_write': is_write,
            'seek_distance': seek_distance,
            'latency_ms': latency
        })
        current_lba = lba
        
    return pd.DataFrame(data)

df = generate_io_traces()
df.head()

## 2. Train ML Latency Predictor
Train a Random Forest regression model to predict the latency of an I/O request based on LBA, seek distance, size, and operation type. This mimics what an in-kernel ML model like KML-IOSched would do.

In [ ]:
# Features: lba, size_kb, is_write, seek_distance
X = df[['lba', 'size_kb', 'is_write', 'seek_distance']]
# Target: latency in milliseconds
y = df['latency_ms']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
print(f"Model Mean Squared Error (MSE): {mse:.4f}")

## 3. ML-Based Scheduling Simulation
We compare a standard First-Come-First-Serve (FCFS) elevator with an ML-Optimized Scheduler. The ML scheduler uses the trained model to predict service times for requests in the queue and greedily picks the one with the lowest predicted latency.

In [ ]:
def simulate_scheduler(requests_df, scheduler_type='FCFS', model=None):
    total_latency = 0
    current_lba = 0
    completed_requests = []
    
    # Copy queue to simulate processing
    queue = requests_df.copy()
    
    while not queue.empty:
        if scheduler_type == 'FCFS':
            # Pop the first request in the queue (chronological)
            idx = queue.index[0]
        elif scheduler_type == 'ML_OPT':
            # Update seek distances from the CURRENT disk head position
            queue['seek_distance'] = abs(queue['lba'] - current_lba)
            
            # Predict latency for all pending items in the queue
            X_queue = queue[['lba', 'size_kb', 'is_write', 'seek_distance']]
            predicted_latencies = model.predict(X_queue)
            
            # Pick the request with the lowest predicted latency (Greedy selection)
            best_loc = np.argmin(predicted_latencies)
            idx = queue.index[best_loc]
            
        req = queue.loc[idx]
        queue = queue.drop(idx)
        
        # Calculate actual latency incurred for this step based on the simulation formula
        actual_seek = abs(req['lba'] - current_lba)
        actual_latency = 2.0 + (actual_seek / 1000000) * 8.0 + (req['size_kb'] / 1024.0) * 1.5
        
        total_latency += actual_latency
        current_lba = req['lba']
        completed_requests.append(actual_latency)
        
    return np.mean(completed_requests)

# Simulate scheduling a batch of 200 random I/O requests
batch = df.sample(200).copy()

fcfs_latency = simulate_scheduler(batch, scheduler_type='FCFS')
ml_latency = simulate_scheduler(batch, scheduler_type='ML_OPT', model=model)

print(f"Average Latency (FCFS Elevator): {fcfs_latency:.2f} ms")
print(f"Average Latency (ML-Optimized Scheduler): {ml_latency:.2f} ms")
print(f"Latency Reduction: {((fcfs_latency - ml_latency) / fcfs_latency) * 100:.2f}%")